# Craft Crosstalk-48k — Stage-2 Chained Full-Bin De-Bleed Refiner (A100 / L4 Auto)
- **Completely isolated** from Phase 1 (`audiosep_48k_v3_86m/`) and Phase 2 (`audiosep_48k_v3_86m_p2/`).
- Freezes your trained `Craft Crosstalk-48k` checkpoint locally in `/content/teacher_best.safetensors` (read-only copy, never locks Phase 2 Drive writes) and trains the **17.95 MB bf16 Full-1,025-Bin 5-Reference Cross-Stem De-Bleed Refiner (v2 Heavy + local-RMS 1D waveform polish)**.
- Auto-tunes batch size & gradient checkpointing for A100-80GB (`~1.5-2.2 hrs`), A100-40GB, or L4-24GB (`~5-7 hrs`).
- Saves checkpoints & live log into `/content/drive/MyDrive/audiosep_48k_refiner/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install soundfile safetensors
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from pathlib import Path

os.makedirs('/content/AudioSep_refiner/refiner', exist_ok=True)
os.makedirs('/content/data48', exist_ok=True)

drive_root = Path('/content/drive/MyDrive/audiosep_data_48k')
p2_code = drive_root / 'p2_code'
ref_code = drive_root / 'refiner_code'

# Copy core dependencies (preferring VRAM-optimized harmonic_slot_tf.py if uploaded in refiner_code)
shutil.copy2(drive_root / 'model.py', '/content/AudioSep_refiner/model.py')
shutil.copy2((p2_code / 'loss.py') if (p2_code / 'loss.py').exists() else (drive_root / 'loss.py'), '/content/AudioSep_refiner/loss.py')
shutil.copy2((p2_code / 'train.py') if (p2_code / 'train.py').exists() else (drive_root / 'train.py'), '/content/AudioSep_refiner/train.py')
for name in ('refiner_model.py', 'train_refiner.py'):
    assert (ref_code / name).exists(), f'Upload {name} to {ref_code} first!'
    shutil.copy2(ref_code / name, f'/content/AudioSep_refiner/refiner/{name}')
if (ref_code / 'harmonic_slot_tf.py').exists():
    shutil.copy2(ref_code / 'harmonic_slot_tf.py', '/content/AudioSep_refiner/model.py')
    print('Using 4.4x VRAM-optimized harmonic_slot_tf.py for frozen teacher!')

!sha256sum /content/AudioSep_refiner/refiner/refiner_model.py /content/AudioSep_refiner/refiner/train_refiner.py /content/AudioSep_refiner/model.py

# Copy frozen teacher snapshot locally (prefers Phase-2 best, falls back to Phase-1 best)
p2_best = Path('/content/drive/MyDrive/audiosep_48k_v3_86m_p2/best.safetensors')
p1_best = Path('/content/drive/MyDrive/audiosep_48k_v3_86m/best.safetensors')
teacher_src = p2_best if p2_best.exists() else p1_best
assert teacher_src.exists(), 'No teacher best.safetensors found on Drive!'
shutil.copy2(teacher_src, '/content/teacher_best.safetensors')
print('Copied frozen teacher snapshot from:', teacher_src)

# Extract cached 48 kHz datasets into /content/data48
tar_sources = [
    ('speech', ['speech.tar']),
    ('rir', ['rir.tar']),
    ('music', ['music_hq48.tar', 'music.tar']),
    ('amb', ['amb_48k.tar', 'amb.tar']),
]
for sub, candidates in tar_sources:
    picked = next((drive_root / c for c in candidates if (drive_root / c).exists()), None)
    if picked is None:
        continue
    dst_sub = Path('/content/data48') / sub
    if dst_sub.exists() and any(dst_sub.iterdir()):
        continue
    print(f'Extracting {picked.name} -> /content/data48/{sub}...')
    subprocess.run(['tar', '-xf', str(picked), '-C', '/content/data48'], check=True)
for sub in ('speech', 'music', 'amb', 'rir'):
    p = Path('/content/data48') / sub
    print(f'/content/data48/{sub}: {sum(1 for f in p.rglob("*") if f.is_file())} files')

In [ ]:
# Train CrossStemDebleedRefiner (17.95 MB bf16) — auto-tuned for A100-80GB / A100-40GB / L4-24GB
import torch
from pathlib import Path
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
if vram_gb >= 70.0:
    gpu_flags = '--batch-size 4 --grad-accum 1'  # A100-80GB: no grad_ckpt
elif vram_gb >= 35.0:
    gpu_flags = '--batch-size 2 --grad-accum 1'  # A100-40GB: no grad_ckpt (~24-28 GB VRAM, ~4x faster!)
else:
    gpu_flags = '--batch-size 1 --grad-accum 2 --grad-checkpoint'  # L4-24GB (~10 GB VRAM)
save_dir = Path('/content/drive/MyDrive/audiosep_48k_refiner')
save_dir.mkdir(parents=True, exist_ok=True)
rflag = f'--resume {save_dir / "last_refiner.pt"}' if (save_dir / 'last_refiner.pt').exists() else ''
print(f'Detected VRAM: {vram_gb:.1f} GB | GPU flags: {gpu_flags} | Resume: {rflag or "(fresh step 0)"}')
!python -u /content/AudioSep_refiner/refiner/train_refiner.py \
    --teacher-ckpt /content/teacher_best.safetensors \
    --precision bf16 --save-bf16 {gpu_flags} \
    --voice-base-c 48 --bg-base-c 32 --voice-bottleneck 3 --bg-bottleneck 2 --wave-hidden 32 \
    --seg-seconds 3.0 --val-seg-seconds 4.0 \
    --p-3spk 0.55 --hard-lang-prob 0.45 --full-overlap-prob 0.55 --mono-mix-prob 0.50 \
    --bleed-weight 3.0 --tf-bleed-weight 1.0 --cross-bleed-weight 2.5 --bg-voice-bleed-weight 1.5 --mix-cons-weight 0.1 \
    --lr 1.8e-4 --ema-decay 0.999 --steps 6000 --eval-every 500 --num-workers 4 \
    --speech-dir /content/data48/speech --music-dir /content/data48/music \
    --amb-dir /content/data48/amb --rir-dir /content/data48/rir \
    --save-dir /content/drive/MyDrive/audiosep_48k_refiner {rflag} 2>&1 | tee -a /content/drive/MyDrive/audiosep_48k_refiner/train_refiner.log